In [ ]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 1: Data Cleaning & Exploratory Data Analysis
==================================================================
Purpose: Load, validate, clean, and explore the QuickBite dataset.
This notebook establishes the foundation for all subsequent analysis.

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

---------------------------------------------------------------------
1. DATA LOADING
---------------------------------------------------------------------

In [ ]:
print("="*80)
print("QUICKBITE DATA LOADING & INITIAL VALIDATION")
print("="*80)

In [ ]:
# Load all datasets
data_path = "../data/"  # Adjust path as needed

In [ ]:
print("\n📂 Loading datasets...")

In [ ]:
cities = pd.read_csv(f"{data_path}cities.csv")
categories = pd.read_csv(f"{data_path}restaurant_categories.csv")
users = pd.read_csv(f"{data_path}users.csv")
restaurants = pd.read_csv(f"{data_path}restaurants.csv")
partners = pd.read_csv(f"{data_path}delivery_partners.csv")
coupons = pd.read_csv(f"{data_path}coupons.csv")
orders = pd.read_csv(f"{data_path}orders.csv")
order_items = pd.read_csv(f"{data_path}order_items.csv")
payments = pd.read_csv(f"{data_path}payments.csv")
weather = pd.read_csv(f"{data_path}weather.csv")
traffic = pd.read_csv(f"{data_path}traffic.csv")

In [ ]:
print(f"✅ Loaded {len(cities)} cities")
print(f"✅ Loaded {len(categories)} restaurant categories")
print(f"✅ Loaded {len(users):,} users")
print(f"✅ Loaded {len(restaurants):,} restaurants")
print(f"✅ Loaded {len(partners):,} delivery partners")
print(f"✅ Loaded {len(coupons)} coupons")
print(f"✅ Loaded {len(orders):,} orders")
print(f"✅ Loaded {len(order_items):,} order items")
print(f"✅ Loaded {len(payments):,} payments")
print(f"✅ Loaded {len(weather):,} weather records")
print(f"✅ Loaded {len(traffic):,} traffic records")

In [ ]:
# Display first few rows of key tables
print("\n" + "="*80)
print("SAMPLE DATA PREVIEW")
print("="*80)

In [ ]:
print("\n📋 Users sample:")
print(users.head(3))

In [ ]:
print("\n📋 Orders sample:")
print(orders.head(3))

In [ ]:
print("\n📋 Order Items sample:")
print(order_items.head(3))

---------------------------------------------------------------------
2. DATA QUALITY VALIDATION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("DATA QUALITY VALIDATION")
print("="*80)

In [ ]:
def validate_table(df, name):
    """Comprehensive data quality check for each table"""
    print(f"\n📊 {name.upper()} Quality Check:")
    print(f"  • Rows: {len(df):,}")
    print(f"  • Columns: {len(df.columns)}")
    print(f"  • Missing values: {df.isnull().sum().sum():,} ({df.isnull().sum().sum()/len(df)*100:.2f}%)")
    
    # Missing values per column
    missing_cols = df.isnull().sum()
    missing_cols = missing_cols[missing_cols > 0]
    if len(missing_cols) > 0:
        print(f"  • Columns with missing data: {list(missing_cols.index)}")
    
    # Duplicate rows
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"  ⚠️ Duplicate rows: {duplicates:,}")
    
    # Data types
    print(f"  • Data types: {df.dtypes.value_counts().to_dict()}")
    
    # Unique values in categorical columns
    for col in df.select_dtypes(include=['object']).columns:
        if len(df[col].unique()) < 20:
            print(f"  • {col}: {df[col].unique().tolist()}")
    
    return missing_cols

In [ ]:
# Validate each table
missing_orders = validate_table(orders, "orders")
missing_users = validate_table(users, "users")
missing_restaurants = validate_table(restaurants, "restaurants")
missing_partners = validate_table(partners, "delivery partners")

---------------------------------------------------------------------
3. DATA CLEANING
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("DATA CLEANING")
print("="*80)

In [ ]:
# 3.1 Convert date columns to datetime
print("\n🔄 Converting date columns...")

In [ ]:
date_columns = {
    'users': ['signup_date', 'premium_start_date', 'churned_at'],
    'orders': ['order_placed_at', 'order_accepted_at', 'food_ready_at', 
               'delivery_partner_assigned_at', 'picked_up_at', 'delivered_at'],
    'restaurants': ['onboarded_date'],
    'partners': ['joined_date'],
    'weather': ['date'],
    'traffic': ['date'],
}

In [ ]:
for table_name, cols in date_columns.items():
    df = eval(table_name)
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f"  ✅ {table_name}: dates converted")

3.2 Handle missing values in orders

In [ ]:
print("\n🔧 Handling missing values...")

In [ ]:
# Orders - keep track of missing before/after
orders_before = len(orders)

In [ ]:
# Remove orders with no order_placed_at (shouldn't happen)
orders = orders.dropna(subset=['order_placed_at'])
print(f"  • Removed {orders_before - len(orders)} orders with missing order_placed_at")

In [ ]:
# For delivered orders, ensure delivered_at exists
delivered_orders = orders[orders['order_status'] == 'delivered']
missing_delivery = delivered_orders['delivered_at'].isnull().sum()
if missing_delivery > 0:
    print(f"  ⚠️ {missing_delivery} delivered orders missing delivered_at - investigating...")
    # These might be orders that were delivered but not logged properly
    # For analysis, we'll keep them but flag them

3.3 Create derived columns

In [ ]:
print("\n📊 Creating derived columns...")

In [ ]:
# For orders: calculate delivery time in minutes for delivered orders
orders['delivery_time_minutes'] = np.where(
    orders['order_status'] == 'delivered',
    (orders['delivered_at'] - orders['order_placed_at']).dt.total_seconds() / 60,
    np.nan
)

In [ ]:
# For orders: calculate prep time in minutes
orders['prep_time_minutes'] = np.where(
    orders['order_status'] == 'delivered',
    (orders['food_ready_at'] - orders['order_accepted_at']).dt.total_seconds() / 60,
    np.nan
)

In [ ]:
# For users: calculate tenure in days
users['tenure_days'] = (pd.Timestamp.now() - users['signup_date']).dt.days

In [ ]:
# For users: calculate age of user (in days since signup)
users['signup_age_days'] = (pd.Timestamp.now() - users['signup_date']).dt.days

In [ ]:
# For users: flag if user has ordered ever
users['has_ordered'] = users['user_id'].isin(orders['user_id'].unique())

In [ ]:
print(f"  ✅ Added delivery_time_minutes to orders")
print(f"  ✅ Added prep_time_minutes to orders")
print(f"  ✅ Added tenure_days to users")
print(f"  ✅ Added has_ordered flag to users")

3.4 Validate business logic in synthetic data

In [ ]:
print("\n🔍 Validating business logic...")

In [ ]:
# Check: Premium users should have higher AOV
premium_aov = orders.merge(users[['user_id', 'is_premium_member']], on='user_id')
premium_aov = premium_aov[premium_aov['order_status'] == 'delivered'].groupby('is_premium_member')['total_amount'].mean()
print(f"  • Premium AOV: ₹{premium_aov.get(True, 0):.2f}")
print(f"  • Non-premium AOV: ₹{premium_aov.get(False, 0):.2f}")

In [ ]:
# Check: Referral users should have better retention
referral_users = users[users['acquisition_channel'] == 'referral']['user_id'].unique()
non_referral_users = users[users['acquisition_channel'] != 'referral']['user_id'].unique()

In [ ]:
referral_orders = orders[orders['user_id'].isin(referral_users)]
non_referral_orders = orders[orders['user_id'].isin(non_referral_users)]

In [ ]:
referral_avg_orders = len(referral_orders) / len(referral_users) if len(referral_users) > 0 else 0
non_referral_avg_orders = len(non_referral_orders) / len(non_referral_users) if len(non_referral_users) > 0 else 0

In [ ]:
print(f"  • Referral users avg orders: {referral_avg_orders:.2f}")
print(f"  • Non-referral avg orders: {non_referral_avg_orders:.2f}")

In [ ]:
# Check: Weather should impact cancellations
weather_cancel = orders.groupby('weather_condition').apply(
    lambda x: (x['order_status'] == 'cancelled').mean()
)
print(f"  • Cancellation rates by weather:")
for condition, rate in weather_cancel.items():
    print(f"    - {condition}: {rate*100:.1f}%")

---------------------------------------------------------------------
4. EXPLORATORY DATA ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPLORATORY DATA ANALYSIS")
print("="*80)

4.1 Orders Overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('QuickBite Orders - Exploratory Analysis', fontsize=16, fontweight='bold')

In [ ]:
# 4.1.1 Order Status Distribution
status_counts = orders['order_status'].value_counts()
ax = axes[0, 0]
status_counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c', '#f39c12'])
ax.set_title('Order Status Distribution')
ax.set_xlabel('Order Status')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
for i, v in enumerate(status_counts.values):
    ax.text(i, v + len(orders)*0.01, f'{v/len(orders)*100:.1f}%', ha='center')

In [ ]:
# 4.1.2 Orders Over Time
orders['order_date'] = orders['order_placed_at'].dt.date
daily_orders = orders.groupby('order_date').size()
ax = axes[0, 1]
daily_orders.plot(ax=ax, color='#3498db', linewidth=2)
ax.set_title('Daily Order Volume')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Orders')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# 4.1.3 AOV Distribution
delivered_orders = orders[orders['order_status'] == 'delivered']
ax = axes[0, 2]
delivered_orders['total_amount'].hist(bins=50, ax=ax, color='#9b59b6', edgecolor='black', alpha=0.7)
ax.axvline(delivered_orders['total_amount'].mean(), color='red', linestyle='--', 
           label=f"Mean: ₹{delivered_orders['total_amount'].mean():.2f}")
ax.axvline(delivered_orders['total_amount'].median(), color='green', linestyle='--',
           label=f"Median: ₹{delivered_orders['total_amount'].median():.2f}")
ax.set_title('Order Value Distribution')
ax.set_xlabel('Order Value (₹)')
ax.set_ylabel('Frequency')
ax.legend()

In [ ]:
# 4.1.4 Cancellation Rate by Weather
weather_cancel_df = orders.groupby('weather_condition').apply(
    lambda x: (x['order_status'] == 'cancelled').mean() * 100
).reset_index(name='cancellation_rate')
weather_cancel_df = weather_cancel_df.sort_values('cancellation_rate', ascending=False)
ax = axes[1, 0]
ax.bar(weather_cancel_df['weather_condition'], weather_cancel_df['cancellation_rate'], 
       color=['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c'])
ax.set_title('Cancellation Rate by Weather Condition')
ax.set_xlabel('Weather Condition')
ax.set_ylabel('Cancellation Rate (%)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# 4.1.5 Hourly Order Distribution
orders['hour'] = orders['order_placed_at'].dt.hour
hourly_orders = orders.groupby('hour').size()
ax = axes[1, 1]
hourly_orders.plot(kind='bar', ax=ax, color='#1abc9c')
ax.set_title('Orders by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Number of Orders')

In [ ]:
# 4.1.6 Top Cities by Orders
city_orders = orders.merge(cities[['city_id', 'city_name']], on='city_id')
city_orders = city_orders['city_name'].value_counts().head(10)
ax = axes[1, 2]
city_orders.plot(kind='barh', ax=ax, color='#e74c3c')
ax.set_title('Top 10 Cities by Order Volume')
ax.set_xlabel('Number of Orders')

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/eda_orders_overview.png', dpi=300, bbox_inches='tight')
plt.show()

4.2 Users Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('QuickBite Users - Exploratory Analysis', fontsize=16, fontweight='bold')

In [ ]:
# 4.2.1 Acquisition Channel Distribution
channel_counts = users['acquisition_channel'].value_counts()
ax = axes[0, 0]
channel_counts.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90)
ax.set_title('User Acquisition Channels')
ax.set_ylabel('')

In [ ]:
# 4.2.2 User Signups Over Time
users['signup_month'] = users['signup_date'].dt.to_period('M')
monthly_signups = users.groupby('signup_month').size()
ax = axes[0, 1]
monthly_signups.plot(kind='line', ax=ax, color='#3498db', marker='o', linewidth=2)
ax.set_title('Monthly User Signups')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Signups')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# 4.2.3 Premium vs Non-Premium Distribution
premium_counts = users['is_premium_member'].value_counts()
ax = axes[0, 2]
colors = ['#e74c3c' if x else '#2ecc71' for x in premium_counts.index]
ax.bar(['Premium', 'Non-Premium'], premium_counts.values, color=colors, alpha=0.7)
ax.set_title('Premium vs Non-Premium Users')
ax.set_ylabel('Number of Users')
for i, v in enumerate(premium_counts.values):
    ax.text(i, v + len(users)*0.01, f'{v/len(users)*100:.1f}%', ha='center')

In [ ]:
# 4.2.4 Age Band Distribution
age_counts = users['age_band'].value_counts()
ax = axes[1, 0]
age_counts.plot(kind='bar', ax=ax, color='#9b59b6', alpha=0.7)
ax.set_title('User Age Distribution')
ax.set_xlabel('Age Band')
ax.set_ylabel('Number of Users')

In [ ]:
# 4.2.5 Device Type Distribution
device_counts = users['device_type'].value_counts()
ax = axes[1, 1]
device_counts.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90)
ax.set_title('Device Distribution')
ax.set_ylabel('')

In [ ]:
# 4.2.6 Orders per User Distribution
user_order_counts = orders.groupby('user_id').size()
user_order_counts = user_order_counts.clip(upper=50)  # Cap for visualization
ax = axes[1, 2]
user_order_counts.hist(bins=50, ax=ax, color='#f39c12', edgecolor='black', alpha=0.7)
ax.axvline(user_order_counts.mean(), color='red', linestyle='--', 
           label=f"Mean: {user_order_counts.mean():.2f}")
ax.set_title('Orders per User Distribution')
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Number of Users')
ax.legend()

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/eda_users_overview.png', dpi=300, bbox_inches='tight')
plt.show()

4.3 Delivery Performance Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('QuickBite Delivery Performance', fontsize=16, fontweight='bold')

In [ ]:
# 4.3.1 Delivery Time Distribution
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()
delivered_orders = delivered_orders.dropna(subset=['delivery_time_minutes'])
delivery_times = delivered_orders['delivery_time_minutes']

In [ ]:
ax = axes[0, 0]
delivery_times.hist(bins=50, ax=ax, color='#3498db', edgecolor='black', alpha=0.7)
ax.axvline(delivery_times.quantile(0.50), color='green', linestyle='--', 
           label=f"P50: {delivery_times.quantile(0.50):.1f} min")
ax.axvline(delivery_times.quantile(0.90), color='orange', linestyle='--',
           label=f"P90: {delivery_times.quantile(0.90):.1f} min")
ax.axvline(delivery_times.quantile(0.99), color='red', linestyle='--',
           label=f"P99: {delivery_times.quantile(0.99):.1f} min")
ax.set_title('Delivery Time Distribution')
ax.set_xlabel('Delivery Time (minutes)')
ax.set_ylabel('Frequency')
ax.legend()

In [ ]:
# 4.3.2 Delivery Time by City
city_delivery = delivered_orders.merge(cities[['city_id', 'city_name']], on='city_id')
city_delivery = city_delivery.groupby('city_name')['delivery_time_minutes'].agg(['mean', 'median'])
city_delivery = city_delivery.sort_values('mean', ascending=False)

In [ ]:
ax = axes[0, 1]
city_delivery[['mean', 'median']].plot(kind='bar', ax=ax, color=['#e74c3c', '#3498db'], alpha=0.7)
ax.set_title('Average Delivery Time by City')
ax.set_xlabel('City')
ax.set_ylabel('Delivery Time (minutes)')
ax.tick_params(axis='x', rotation=45)
ax.legend(['Mean', 'Median'])

In [ ]:
# 4.3.3 Delivery Time vs Distance
ax = axes[1, 0]
ax.scatter(delivered_orders['distance_km'], delivered_orders['delivery_time_minutes'], 
          alpha=0.3, color='#2ecc71')
ax.set_xlabel('Distance (km)')
ax.set_ylabel('Delivery Time (minutes)')
ax.set_title('Delivery Time vs Distance')

In [ ]:
# Add correlation
corr = delivered_orders['distance_km'].corr(delivered_orders['delivery_time_minutes'])
ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes, 
        bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))

In [ ]:
# 4.3.4 Prep Time vs Total Delivery Time
ax = axes[1, 1]
ax.scatter(delivered_orders['prep_time_minutes'], delivered_orders['delivery_time_minutes'], 
          alpha=0.3, color='#9b59b6')
ax.set_xlabel('Prep Time (minutes)')
ax.set_ylabel('Total Delivery Time (minutes)')
ax.set_title('Prep Time vs Total Delivery Time')

In [ ]:
corr = delivered_orders['prep_time_minutes'].corr(delivered_orders['delivery_time_minutes'])
ax.text(0.05, 0.95, f'Correlation: {corr:.2f}', transform=ax.transAxes,
        bbox=dict(boxstyle="round", facecolor='white', alpha=0.8))

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/eda_delivery_performance.png', dpi=300, bbox_inches='tight')
plt.show()

4.4 Revenue Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('QuickBite Revenue Analysis', fontsize=16, fontweight='bold')

In [ ]:
# 4.4.1 Monthly GMV Trend
orders['order_month'] = orders['order_placed_at'].dt.to_period('M')
monthly_gmv = orders[orders['order_status'] == 'delivered'].groupby('order_month')['total_amount'].sum()

In [ ]:
ax = axes[0, 0]
monthly_gmv.plot(kind='line', ax=ax, color='#2ecc71', marker='o', linewidth=2)
ax.set_title('Monthly GMV Trend')
ax.set_xlabel('Month')
ax.set_ylabel('GMV (₹)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# 4.4.2 AOV by City
city_aov = orders[orders['order_status'] == 'delivered'].merge(
    cities[['city_id', 'city_name']], on='city_id'
).groupby('city_name')['total_amount'].mean().sort_values(ascending=False)

In [ ]:
ax = axes[0, 1]
city_aov.plot(kind='barh', ax=ax, color='#e67e22')
ax.set_title('Average Order Value by City')
ax.set_xlabel('AOV (₹)')

In [ ]:
# 4.4.3 Revenue by Acquisition Channel
channel_revenue = orders[orders['order_status'] == 'delivered'].merge(
    users[['user_id', 'acquisition_channel']], on='user_id'
).groupby('acquisition_channel')['total_amount'].sum().sort_values(ascending=False)

In [ ]:
ax = axes[1, 0]
channel_revenue.plot(kind='bar', ax=ax, color='#3498db')
ax.set_title('Revenue by Acquisition Channel')
ax.set_xlabel('Channel')
ax.set_ylabel('Total Revenue (₹)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# 4.4.4 Coupon Performance
coupon_orders = orders[orders['coupon_id'].notna()]
coupon_performance = coupon_orders.merge(
    coupons[['coupon_id', 'coupon_code', 'discount_type']], on='coupon_id'
).groupby('coupon_code').agg({
    'order_id': 'count',
    'discount_amount': 'sum',
    'total_amount': 'sum'
}).reset_index()
coupon_performance['revenue_per_discount'] = coupon_performance['total_amount'] / coupon_performance['discount_amount']

In [ ]:
ax = axes[1, 1]
coupon_performance_sorted = coupon_performance.sort_values('revenue_per_discount', ascending=False)
ax.barh(coupon_performance_sorted['coupon_code'], coupon_performance_sorted['revenue_per_discount'], 
        color='#e74c3c')
ax.set_title('Revenue per Discount Rupee by Coupon')
ax.set_xlabel('Revenue per ₹1 Discount')

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/eda_revenue_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
5. CORRELATION ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("CORRELATION ANALYSIS")
print("="*80)

5.1 Feature correlation for key numeric variables

In [ ]:
# Prepare a dataset for correlation analysis
corr_data = orders[orders['order_status'] == 'delivered'].copy()

In [ ]:
# Add user-level features
corr_data = corr_data.merge(
    users[['user_id', 'is_premium_member']], on='user_id', how='left'
)

In [ ]:
# Select numeric columns for correlation
numeric_cols = [
    'total_amount', 'subtotal_amount', 'delivery_fee', 'discount_amount',
    'distance_km', 'traffic_index_at_order', 'delivery_time_minutes', 'prep_time_minutes'
]

In [ ]:
# Add boolean columns as numeric
corr_data['is_premium'] = corr_data['is_premium_member'].astype(int)

In [ ]:
# Calculate correlation matrix
corr_matrix = corr_data[numeric_cols + ['is_premium']].corr()

In [ ]:
# Create heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of Key Business Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/eda_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 5.2 Key correlation findings
print("\n📊 Key Correlation Insights:")
print("-" * 50)

In [ ]:
# Find strongest correlations
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1]  # Remove self-correlations

In [ ]:
print("\nStrongest positive correlations:")
for pair, value in corr_pairs.head(5).items():
    print(f"  • {pair[0]} ↔ {pair[1]}: {value:.2f}")

In [ ]:
print("\nStrongest negative correlations:")
for pair, value in corr_pairs.tail(5).items():
    print(f"  • {pair[0]} ↔ {pair[1]}: {value:.2f}")

---------------------------------------------------------------------
6. DATA QUALITY SUMMARY REPORT
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("DATA QUALITY SUMMARY REPORT")
print("="*80)

In [ ]:
# Create a comprehensive data quality report
quality_report = {
    'Table': [],
    'Rows': [],
    'Columns': [],
    'Missing_Values': [],
    'Missing_Percent': [],
    'Duplicates': [],
    'Unique_Users': []
}

In [ ]:
tables = ['cities', 'categories', 'users', 'restaurants', 'partners', 
          'coupons', 'orders', 'order_items', 'payments', 'weather', 'traffic']

In [ ]:
for table in tables:
    df = eval(table)
    quality_report['Table'].append(table)
    quality_report['Rows'].append(len(df))
    quality_report['Columns'].append(len(df.columns))
    missing = df.isnull().sum().sum()
    quality_report['Missing_Values'].append(missing)
    quality_report['Missing_Percent'].append(missing / (len(df) * len(df.columns)) * 100)
    quality_report['Duplicates'].append(df.duplicated().sum())
    
    if 'user_id' in df.columns:
        quality_report['Unique_Users'].append(df['user_id'].nunique())
    else:
        quality_report['Unique_Users'].append(None)

In [ ]:
quality_df = pd.DataFrame(quality_report)
print("\n" + quality_df.to_string())

---------------------------------------------------------------------
7. SAVE CLEANED DATA
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SAVING CLEANED DATA")
print("="*80)

In [ ]:
# Create output directory if it doesn't exist
import os
os.makedirs('../outputs/cleaned_data', exist_ok=True)
os.makedirs('../outputs/visualizations', exist_ok=True)

In [ ]:
# Save cleaned datasets with derived columns
users.to_csv('../outputs/cleaned_data/users_cleaned.csv', index=False)
orders.to_csv('../outputs/cleaned_data/orders_cleaned.csv', index=False)
restaurants.to_csv('../outputs/cleaned_data/restaurants_cleaned.csv', index=False)
partners.to_csv('../outputs/cleaned_data/partners_cleaned.csv', index=False)
order_items.to_csv('../outputs/cleaned_data/order_items_cleaned.csv', index=False)
payments.to_csv('../outputs/cleaned_data/payments_cleaned.csv', index=False)

In [ ]:
print("✅ Cleaned datasets saved to ../outputs/cleaned_data/")
print("✅ Visualizations saved to ../outputs/visualizations/")

---------------------------------------------------------------------
8. SUMMARY STATISTICS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("KEY BUSINESS STATISTICS")
print("="*80)

In [ ]:
# Calculate key metrics
total_orders = len(orders)
delivered_orders = len(orders[orders['order_status'] == 'delivered'])
cancelled_orders = len(orders[orders['order_status'] == 'cancelled'])
total_users = len(users)
active_users = len(users[users['has_ordered']])
total_gmv = orders[orders['order_status'] == 'delivered']['total_amount'].sum()
avg_aov = orders[orders['order_status'] == 'delivered']['total_amount'].mean()
avg_delivery_time = orders[orders['order_status'] == 'delivered']['delivery_time_minutes'].mean()
p90_delivery_time = orders[orders['order_status'] == 'delivered']['delivery_time_minutes'].quantile(0.90)
p99_delivery_time = orders[orders['order_status'] == 'delivered']['delivery_time_minutes'].quantile(0.99)
premium_users = len(users[users['is_premium_member'] == True])

In [ ]:
print(f"""
📊 QUICKBITE PLATFORM - KEY METRICS
====================================
Total Orders:           {total_orders:,}
  • Delivered:          {delivered_orders:,} ({delivered_orders/total_orders*100:.1f}%)
  • Cancelled:          {cancelled_orders:,} ({cancelled_orders/total_orders*100:.1f}%)

Total Users:            {total_users:,}
  • Active Users:       {active_users:,} ({active_users/total_users*100:.1f}%)
  • Premium Users:      {premium_users:,} ({premium_users/total_users*100:.1f}%)

GMV:                    ₹{total_gmv:,.2f}
Average Order Value:    ₹{avg_aov:.2f}

Delivery Performance:
  • Average Time:       {avg_delivery_time:.1f} minutes
  • P90 Time:           {p90_delivery_time:.1f} minutes
  • P99 Time:           {p99_delivery_time:.1f} minutes

Repeat Purchase Rate:   {len(orders[orders.duplicated('user_id', keep=False)]) / len(orders['user_id'].unique())*100:.1f}%
""")

In [ ]:
print("\n" + "="*80)
print("✅ DATA CLEANING & EDA COMPLETE")
print("="*80)
print("\n📌 Next Steps:")
print("  1. Proceed to Notebook 2: Cohort Retention Analysis")
print("  2. Review visualizations saved in /outputs/")
print("  3. Validate business logic findings")
print("="*80)